# 01 · Understand the sales data

Start with the grain and data quality before modeling. Source: [dunnhumby Breakfast at the Frat](https://www.dunnhumby.com/source-files/), a representation of historical retail data. Run the setup command in the README first.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").exists() else Path.cwd().parent
from retail_pricing.config import DATA, ARTIFACTS, REPORTS
observations = pd.read_parquet(DATA / "processed/observations.parquet")
products = pd.read_parquet(DATA / "processed/products.parquet")
quality = json.loads((REPORTS / "data_quality.json").read_text())


## Coverage and rejected observations
Missing weeks are not zero sales. Lookup counts can differ from observed counts. The source guide and workbook are preserved under `data/raw`.

In [2]:
display(pd.Series(quality).to_frame("value"))
rejected = pd.read_parquet(DATA / "processed/rejected.parquet")
display(rejected[["price", "base_price", "units", "reason"]].head(10))

,value
source_rows,524950
clean_rows,524741
rejected_rows,209
rejection_reasons,"{'invalid_price': 24, 'invalid_base_price': 185}"
zero_sales_retained,0
products,55
stores,77
weeks,156
first_week,2009-01-14
last_week,2012-01-04


,price,base_price,units,reason
0,3.32,NaN,56,invalid_base_price;
1,3.32,NaN,38,invalid_base_price;
2,3.31,NaN,72,invalid_base_price;
3,3.33,NaN,52,invalid_base_price;
4,3.39,NaN,52,invalid_base_price;
5,3.30,NaN,52,invalid_base_price;
6,3.32,NaN,46,invalid_base_price;
7,3.28,NaN,82,invalid_base_price;
8,3.33,NaN,28,invalid_base_price;
9,2.50,NaN,2,invalid_base_price;


## Which products dominate revenue?
Use source spend for historical revenue. The simulator later calculates price × predicted units.

In [3]:
ranking = observations.groupby("product_id").agg(units=("units", "sum"), revenue=("revenue", "sum"), rows=("units", "size")).reset_index().merge(products, on="product_id")
display(ranking.sort_values("revenue", ascending=False)[["description", "category", "units", "revenue", "rows"]].head(10))

,description,category,units,revenue,rows
12,GM HONEY NUT CHEERIOS,COLD CEREAL,804228,2048896.22,11985
32,KELL FROSTED FLAKES,COLD CEREAL,565574,1483345.12,11979
45,DIGRN PEPP PIZZA,FROZEN PIZZA,259557,1437337.39,11961
13,GM CHEERIOS,COLD CEREAL,367984,1433001.72,11983
14,GM CHEERIOS,COLD CEREAL,517126,1384585.78,11980
31,KELL BITE SIZE MINI WHEAT,COLD CEREAL,396439,1216282.24,11904
33,KELL FROOT LOOPS,COLD CEREAL,458482,1150913.50,11947
44,DIGRN SUPREME PIZZA,FROZEN PIZZA,191481,1055817.38,11918
8,PL BT SZ FRSTD SHRD WHT,COLD CEREAL,468195,991368.53,11963
52,POST HNY BN OTS HNY RSTD,COLD CEREAL,304234,954418.23,10700


In [4]:
weekly = observations.groupby("week", as_index=False).agg(units=("units", "sum"), revenue=("revenue", "sum"))
px.line(weekly, x="week", y="units", title="Observed weekly units across the dataset").show(renderer="plotly_mimetype")

## Promotion coverage and unusual purchases
Do not silently trim unusually large purchases. Units per visit can identify observations worth inspecting; without inventory data we cannot label low sales as stockouts.

In [5]:
display(observations.groupby(["feature", "display", "tpr_only"]).size().rename("rows").reset_index())
ratios = observations.assign(units_per_visit=observations.units / observations.visits, visits_per_household=observations.visits / observations.hhs)
display(ratios.nlargest(10, "units_per_visit")[["product_id", "store_id", "week", "units", "visits", "units_per_visit"]])

,feature,display,tpr_only,rows
0,0,0,0,375541
1,0,0,1,70733
2,0,1,0,34291
3,1,0,0,20800
4,1,1,0,23376


,product_id,store_id,week,units,visits,units_per_visit
299615,3700031613,6179,2010-11-24,51,2,25.500000
1164,1111009477,13827,2010-12-01,480,25,19.200000
1163,1111009477,13827,2010-11-24,670,39,17.179487
1154,1111009477,13827,2010-09-22,422,36,11.722222
1165,1111009477,13827,2010-12-08,519,45,11.533333
70150,1111085319,19521,2011-10-05,102,9,11.333333
1161,1111009477,13827,2010-11-10,315,28,11.250000
187089,2840002333,17599,2009-02-18,22,2,11.000000
1187,1111009477,13827,2011-05-11,175,16,10.937500
1160,1111009477,13827,2010-11-03,318,30,10.600000


## Takeaway
The data contains useful price and promotion variation but not randomized interventions or inventory. Keep source anomalies explicit, and split on time before fitting transformations. See `docs/data.md` for the full cleaning policy.